# Model Persistence

Save and reload trained model for reuse in production and simulation.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ModelPersistence") \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 17:45:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Load Training Dataset

In [2]:
train_df = spark.read.parquet("../data/processed/train")

### 3. Train Final Model

In [3]:
from pyspark.sql.functions import col, when
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# compute weights
fraud_count = train_df.filter(col("label") == 1).count()
legit_count = train_df.filter(col("label") == 0).count()

total = fraud_count + legit_count

fraud_weight = total / (2 * fraud_count)
legit_weight = total / (2 * legit_count)

# add weight column
train_weighted = train_df.withColumn(
    "weight",
    when(col("label") == 1, fraud_weight).otherwise(legit_weight)
)

# model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=20,
    maxDepth=5,
    seed=42
)

pipeline = Pipeline(stages=[rf])
model = pipeline.fit(train_weighted)

print("Final model trained")

26/05/13 17:45:57 WARN MemoryStore: Not enough space to cache rdd_44_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:45:57 WARN BlockManager: Persisting block rdd_44_10 to disk instead.
26/05/13 17:45:57 WARN MemoryStore: Not enough space to cache rdd_44_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:45:57 WARN BlockManager: Persisting block rdd_44_3 to disk instead.
26/05/13 17:46:04 WARN MemoryStore: Not enough space to cache rdd_44_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:46:04 WARN MemoryStore: Not enough space to cache rdd_44_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:46:07 WARN MemoryStore: Not enough space to cache rdd_44_10 in memory! (computed 211.0 MiB so far)
26/05/13 17:46:07 WARN MemoryStore: Not enough space to cache rdd_44_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:46:12 WARN MemoryStore: Not enough space to cache rdd_44_3 in memory! (computed 211.0 MiB so far)
26/05/13 17:46:12 WARN MemoryStore: Not enough space to cache rdd_

Final model trained


### 4. Save Model

In [4]:
model.write().overwrite().save("../models/fraud_model")

print("Model saved to /models/fraud_model")

Model saved to /models/fraud_model


### 5. Load Model

In [5]:
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load("../models/fraud_model")

print("Model loaded successfully")

Model loaded successfully


### 6. Test Prediction (!)

In [6]:
test_df = spark.read.parquet("../data/processed/test")

predictions = loaded_model.transform(test_df)

predictions.select("features", "prediction").show(5)

[Stage 46:===========================================>              (3 + 1) / 4]

+--------------------+----------+
|            features|prediction|
+--------------------+----------+
|(10,[0,1,2,5,8],[...|       0.0|
|(10,[0,1,2,5,8],[...|       0.0|
|(10,[0,1,2,5,8],[...|       0.0|
|(10,[0,1,2,5,8],[...|       0.0|
|(10,[0,1,2,5,8],[...|       0.0|
+--------------------+----------+
only showing top 5 rows



In [7]:
predictions.groupBy("prediction").count().show()

[Stage 47:================================================>       (13 + 2) / 15]

+----------+-------+
|prediction|  count|
+----------+-------+
|       0.0|1273088|
|       1.0|   1622|
+----------+-------+



## Summary

- model saved to /models directory
- model can be reloaded without retraining
- prediction pipeline works correctly after loading

Model persistence confirmed.